# Data analyisis

First we load... 

In [26]:
import pandas as pd
import os

repo_root = os.getcwd()
file_path = os.path.join(repo_root, '..', "Data Collection", "MASTER_ALL_DATA.csv")

raw_df = pd.read_csv(file_path)

# # Add a parameter that tells read_csv to not parse strings as NaN
# df_different_parsing = pd.read_csv(file_path, keep_default_na=False)

raw_df.head(3)

/var/folders/sz/8v4khwpx0b76jkngnc80xll40000gn/T/ipykernel_56095/4190149445.py:7: DtypeWarning: Columns (0: title, 1: created_utc, 2: permalink) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(file_path)


,app_id,game_name,genre,month,peak_players,avg_players,engagement,valence,n_posts,title,score,upvote,upvote_ratio,num_comments,created_utc,permalink
0,1063730,New World: Aeternum,"Action, Adventure, Massively Multiplayer, RPG",Last 30 Days,987,550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1063730,New World: Aeternum,"Action, Adventure, Massively Multiplayer, RPG",April 2026,975,579.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1063730,New World: Aeternum,"Action, Adventure, Massively Multiplayer, RPG",March 2026,"1,093",581.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# NaN Problem & Data format :')
Cause of NaN Problem: we have vertical stack of data, we need to merge the 3 data sources to have a horizontal (full) dataset to work with

--> re-separate 3 sources, then merge using app_id as the merge id

In [7]:
raw_df = pd.read_csv(file_path, dtype={'app_id': int, 'title': str, 'created_utc': str, 'permalink': str})

#Steam Monthly Player Data
steam_df = raw_df[raw_df['month'].notna()].copy()
steam_df = steam_df[['app_id', 'game_name', 'genre', 'month', 'peak_players', 'avg_players']]

# Reddit engagement data 
reddit_engagement = raw_df[raw_df['engagement'].notna()].copy()
reddit_engagement = reddit_engagement[['app_id', 'engagement', 'valence', 'n_posts']]

# Ensure one row per game app_id
reddit_engagement = reddit_engagement.drop_duplicates(subset=['app_id'])

# (Inner Join) merge using app_id, we only keep data where there is a merge partner
analysis_df = pd.merge(steam_df, reddit_engagement, on='app_id', how='inner')

Overview 2.O

In [25]:
#complete rows
print(f"Cleaned Dataset contains {analysis_df.shape[0]} valid rows for analysis.")
print(analysis_df.head(3))


#per game:
nb_games = len(analysis_df['game_name'].unique())

print(f"Cleaned Dataset contains {nb_games} games for analysis.")

analysis_df['game_name'].value_counts()


Cleaned Dataset contains 443820 valid rows for analysis.
    app_id            game_name  \
0  1063730  New World: Aeternum   
1  1063730  New World: Aeternum   
2  1063730  New World: Aeternum   

                                           genre         month peak_players  \
0  Action, Adventure, Massively Multiplayer, RPG  Last 30 Days          987   
1  Action, Adventure, Massively Multiplayer, RPG    April 2026          975   
2  Action, Adventure, Massively Multiplayer, RPG    March 2026        1,093   

  avg_players  engagement  valence  n_posts  
0       550.0         0.0      NaN      0.0  
1       579.3         0.0      NaN      0.0  
2       581.7         0.0      NaN      0.0  
Cleaned Dataset contains 1917 games for analysis.


game_name
Heroes & Generals             1232
TrackMania Nations Forever    1169
theHunter Classic             1152
Transformice                  1096
Clicker Heroes                1064
                              ... 
Pretend                         20
校长模拟器                           11
Dungeon Royale                   6
The Legend of Karl               5
Fantasy Royal VR                 2
Name: count, Length: 1917, dtype: int64

# 1. Gain overview

- see what data we're dealing with
- find missing values (so we can decide how to handle them)
- check completeness
- check distribution of values to see which analyses are viable

In [ ]:
print(f"Full Dataset: Total games: {len(df)}")
df.head()

Full Dataset: Total games: 453427


,app_id,game_name,genre,month,peak_players,avg_players,engagement,valence,n_posts,title,score,upvote,upvote_ratio,num_comments,created_utc,permalink
0,1063730,New World: Aeternum,"Action, Adventure, Massively Multiplayer, RPG",Last 30 Days,987,550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1063730,New World: Aeternum,"Action, Adventure, Massively Multiplayer, RPG",April 2026,975,579.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1063730,New World: Aeternum,"Action, Adventure, Massively Multiplayer, RPG",March 2026,"1,093",581.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1063730,New World: Aeternum,"Action, Adventure, Massively Multiplayer, RPG",February 2026,"1,367",738.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1063730,New World: Aeternum,"Action, Adventure, Massively Multiplayer, RPG",January 2026,"1,703",933.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1063730,New World: Aeternum,"Action, Adventure, Massively Multiplayer, RPG",December 2025,"2,278","1,222.0",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1063730,New World: Aeternum,"Action, Adventure, Massively Multiplayer, RPG",November 2025,"9,582","3,712.2",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1063730,New World: Aeternum,"Action, Adventure, Massively Multiplayer, RPG",October 2025,"50,987","21,286.0",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1063730,New World: Aeternum,"Action, Adventure, Massively Multiplayer, RPG",September 2025,"17,482","9,062.5",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1063730,New World: Aeternum,"Action, Adventure, Massively Multiplayer, RPG",August 2025,"13,351","8,497.6",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 2. Clean up Data
- Commas in peak_players (e.g., "1,093" -> 1093)

In [ ]:
# commas
steam_df['peak_players'] = steam_df['peak_players'].astype(str).str.replace(',', '').astype(float)
steam_df['avg_players'] = steam_df['avg_players'].astype(str).str.replace(',', '').astype(float)



In [4]:
# What would that dropping NaNs do to our data?
df.dropna().info()

df.describe().style

df.info()

# # Define the features we want to change and store this list for later
# categorical_features = []
# # Let's see how many unique values we have for each of these features
# print(df[categorical_features].nunique())

# # Convert string columns to categorical data
# df[categorical_features] = df[categorical_features].astype("category")


# # Make a deep copy of our dataframe to store the original dataframe for later comparison
# df_deepfake = df.copy()

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   app_id        0 non-null      int64  
 1   game_name     0 non-null      str    
 2   genre         0 non-null      str    
 3   month         0 non-null      str    
 4   peak_players  0 non-null      str    
 5   avg_players   0 non-null      str    
 6   engagement    0 non-null      float64
 7   valence       0 non-null      float64
 8   n_posts       0 non-null      float64
 9   title         0 non-null      str    
 10  score         0 non-null      float64
 11  upvote        0 non-null      float64
 12  upvote_ratio  0 non-null      float64
 13  num_comments  0 non-null      float64
 14  created_utc   0 non-null      str    
 15  permalink     0 non-null      str    
dtypes: float64(7), int64(1), str(8)
memory usage: 132.0 bytes
<class 'pandas.DataFrame'>
RangeIndex: 453427 entries, 0 to 453426
Data columns (